# Week 1 Notebook
*--by Sardor*

I downloaded the full dataset from https://github.com/wadefagen/datasets/blob/main/gpa/uiuc-gpa-dataset.csv, loaded it into a Jupyter notebook and analyzed it  via `pandas`.

In [1]:
import pandas as pd

grades = pd.read_csv('../data/uiuc-gpa-dataset.csv')
grades.head()

,Year,Term,YearTerm,Subject,Number,Course Title,Sched Type,A+,A,A-,...,C+,C,C-,D+,D,D-,F,W,Students,Primary Instructor
0,2026,Winter,2026-wi,ADV,150,Introduction to Advertising,ONL,23,2,0,...,0,0,0,0,0,0,0,0,26,"Hall, Steven R"
1,2026,Winter,2026-wi,ANTH,103,Anthro in a Changing World,ONL,38,1,1,...,0,0,0,0,0,0,0,0,42,"Patino, Michelle"
2,2026,Winter,2026-wi,ANTH,246,Forensic Science,ONL,5,18,1,...,0,1,0,0,0,0,1,0,29,"Reich, Arielle Y"
3,2026,Winter,2026-wi,ASTR,330,Extraterrestrial Life,ONL,8,27,0,...,0,0,0,0,0,0,0,0,36,"Looney, Leslie W"
4,2026,Winter,2026-wi,BADM,275,Intro to Op & Supply Chain Mgt,ONL,0,55,5,...,0,0,0,0,0,0,0,0,61,"Sumkin, Dmitrii"


To see if there's one record per course, instructor, term triple I use a groupby and see how many entries there are in each group. If there was one record per that triple, I expect the count to be 1. As we see below, there are entries where count is not 1, which lets us conclude that this dataset has one record per section:

In [2]:
grades.groupby(['YearTerm', 'Subject', 'Number', 'Primary Instructor'])['Year'].count()

YearTerm  Subject  Number  Primary Instructor  
2010-fa   AAS      100     Arnaldo, Constancio     2
                           Kwon, Yaejoon           2
                           Manalansan, Martin F    1
                           Winkelmann, Marie T     1
                   120     Lee, Alexander J        2
                                                  ..
2026-wi   PSYC     204     Dolcos, Sanda M         1
          REL      110     Ebel, Jonathan H        1
          SOC      100     Dill, Brian             1
          SPAN     130     Abbott, Annie R         1
                   142     Dutton, Michelle M      1
Name: Year, Length: 59415, dtype: int64

To see the available terms, we may reference the [data document](https://github.com/wadefagen/datasets/blob/main/gpa/README.md). We can also confirm it for ourselves. I do this by first finding the range of years and terms over which the dataset goes:

In [ ]:
min_year = grades['Year'].min()
print(min_year)
grades.loc[grades['Year'] == min_year, 'Term'].unique()

2010


<StringArray>
['Fall', 'Spring', 'Summer']
Length: 3, dtype: str

In [8]:
max_year = grades['Year'].max()
print(max_year)
grades.loc[grades['Year'] == max_year, 'Term'].unique()

2026


<StringArray>
['Winter']
Length: 1, dtype: str

So the earliest available term is Spring 2010 and latest available term is Winter 2026. To see what terms are missing from the range, we make all possible combinations of the YearTerm and left join this new dataframe on the YearTerm. This way, the missing terms will have their non-YearTerm fields be `NA`, through which we can query them: 

In [9]:
poy_codes = grades['YearTerm'].str.split('-').apply(lambda x: x[1]).unique()

all_terms = pd.DataFrame(
    {
        'YearTerm' : [f'{year}-{poy}' for year in range(min_year, max_year + 1) for poy in poy_codes]
    }
)

In [10]:
all_terms.merge(grades, 'left').query('Year.isna()')['YearTerm'].unique()

<StringArray>
['2010-wi', '2011-wi', '2011-su', '2012-wi', '2012-sp', '2013-wi', '2014-wi',
 '2026-fa', '2026-sp', '2026-su']
Length: 10, dtype: str

So missing terms are Winter terms of 2010, 2011, 2012, 2013, 2014, Summer term of 2011, Spring term of 2012 and all non-winter terms of 2026, which is consistent with the data doc.

Now we count CS rows:

In [11]:
cs = grades.loc[grades['Subject'] == 'CS']
cs.shape[0]

2593